Let's validate a trained checkpoint using an image pairs dataset. WE'll start by defining some variables.

In [ ]:
checkpoint_path = "./checkpoints/checkpoint.pt"
lr_images_path = "./dataset/validate/lr"
hr_images_path = "./dataset/validate/hr"
device = "cpu"

Next, we'll set up the validation set and loader.

In [ ]:
from data import ImagePairs

from torch.utils.data import DataLoader


dataset = ImagePairs(lr_images_path, hr_images_path)

dataloader = DataLoader(
    dataset,
    batch_size=1,
    pin_memory="cuda" in device,
)

Now the model.

In [ ]:
import torch

from src.mewzoom.model import MewZoom


checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

upscaler = MewZoom(**checkpoint["model_args"])

upscaler.model.add_qa_head(checkpoint["degradation_features"])
upscaler.model.add_weight_norms()

state_dict = checkpoint["model"]

# Compensate for compiled state dict.
for key in list(state_dict.keys()):
    state_dict[key.replace("_orig_mod.", "")] = state_dict.pop(key)

upscaler.load_state_dict(state_dict)

upscaler.remove_parameterizations()
upscaler.model.remove_qa_head()

upscaler = upscaler.to(device)

upscaler.eval()

print("Model checkpoint loaded successfully")

Let's get some metrics ready.

In [ ]:
from torchmetrics.image import (
    PeakSignalNoiseRatio,
    StructuralSimilarityIndexMeasure,
    VisualInformationFidelity,
    FrechetInceptionDistance,
)


psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_metric = StructuralSimilarityIndexMeasure().to(device)
vif_metric = VisualInformationFidelity().to(device)
fid_metric = FrechetInceptionDistance(feature=64).to(device)

Gotta get them values some how - let's do that now.

In [ ]:
from tqdm import tqdm


for x, y in tqdm(dataloader, desc="Testing", leave=False):
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)

    y_pred = upscaler.upscale(x)

    psnr_metric.update(y_pred, y)
    ssim_metric.update(y_pred, y)
    vif_metric.update(y_pred, y)

    fid_metric.update(y_pred, real=False)
    fid_metric.update(y, real=True)

psnr = psnr_metric.compute()
ssim = ssim_metric.compute()
vif = vif_metric.compute()
fid = fid_metric.compute()

print(f"PSNR: {psnr:.5f}, SSIM: {ssim:.5f}, VIF: {vif:.5f}, FID: {fid:.5f}")